# Bahdanau Attention from Scratch

Implementing the attention mechanism from "Neural Machine Translation by Jointly Learning to Align and Translate" (2015).

**Key idea:** Instead of compressing the entire input into a single context vector, let the decoder "look back" at all encoder hidden states and focus on relevant ones at each step.

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from collections import Counter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cpu')
print(f'Using device: {device}')

Using device: cpu


## 1. Load Multi30k Dataset

In [6]:
# Load dataset from HuggingFace
dataset = load_dataset('bentrevett/multi30k')

# Use subset for CPU training
TRAIN_SIZE = 5000
VAL_SIZE = 500

train_data = dataset['train'].select(range(TRAIN_SIZE))
val_data = dataset['validation'].select(range(VAL_SIZE))

In [31]:
print(f'Training samples: {len(train_data)}')
print(f'Validation samples: {len(val_data)}')
print()
print('Example:')
print(f"  EN: {train_data[200]['en']}")
print(f"  DE: {train_data[200]['de']}")

Training samples: 5000
Validation samples: 500

Example:
  EN: Woman on a hill by a white cross overlooking a beach.
  DE: Frau auf einem Hügel neben einem weißen Kreuz, die auf einen Strand blickt.


## 2. Tokenization & Vocabulary

In [42]:
def tokenize(text):
    return text.lower().strip().split()

class Vocabulary:
    def __init__(self, name, min_freq=2):
        self.name = name
        self.min_freq = min_freq
        self.word2idx = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
        self.idx2word = {0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>'}
        self.n_words = 4
    
    def build_vocab(self, sentences):
        counter = Counter()
        for sent in sentences:
            counter.update(tokenize(sent))     
        
        for word, freq in counter.items():
            if freq >= self.min_freq:
                self.word2idx[word] = self.n_words
                self.idx2word[self.n_words] = word
                self.n_words += 1
    
    def encode(self, sentence):
        tokens = tokenize(sentence)
        return [self.word2idx.get(w, self.word2idx['<UNK>']) for w in tokens]
    
    def decode(self, indices):
        return [self.idx2word[idx] for idx in indices if idx not in [0, 1, 2]]

# Build vocabularies
src_vocab = Vocabulary('english', min_freq=2)
trg_vocab = Vocabulary('german', min_freq=2)

src_vocab.build_vocab([ex['en'] for ex in train_data])
trg_vocab.build_vocab([ex['de'] for ex in train_data])

print(f'English vocabulary: {src_vocab.n_words} words')
print(f'German vocabulary: {trg_vocab.n_words} words')

English vocabulary: 2687 words
German vocabulary: 2773 words


## 3. Dataset & DataLoader

In [58]:
class TranslationDataset(torch.utils.data.Dataset):
    def __init__(self, data, src_vocab, trg_vocab):
        self.data = data
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab
    
    def __len__(self):
        return len(self.data)  
    
    def __getitem__(self, idx):
        src = self.src_vocab.encode(self.data[idx]['en']) + [self.src_vocab.word2idx['<EOS>']]
        trg = [self.trg_vocab.word2idx['<SOS>']] + self.trg_vocab.encode(self.data[idx]['de']) + [self.trg_vocab.word2idx['<EOS>']]
        return torch.tensor(src), torch.tensor(trg)

def collate_fn(batch):
    src_batch, trg_batch = zip(*batch)
    src_padded = nn.utils.rnn.pad_sequence(src_batch, padding_value=0)
    trg_padded = nn.utils.rnn.pad_sequence(trg_batch, padding_value=0)
    return src_padded, trg_padded

# Create datasets
train_dataset = TranslationDataset(train_data, src_vocab, trg_vocab)
val_dataset = TranslationDataset(val_data, src_vocab, trg_vocab)

# Create dataloaders
BATCH_SIZE = 32

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)

# Test
src, trg = next(iter(train_loader))
print(f'Source batch shape: {src.shape}  # [src_len, batch]')
print(f'Target batch shape: {trg.shape}  # [trg_len, batch]')

src_vocab.decode((src[:, 10]).tolist())

Source batch shape: torch.Size([21, 32])  # [src_len, batch]
Target batch shape: torch.Size([20, 32])  # [trg_len, batch]


['a',
 'woman',
 'is',
 'reading',
 'a',
 'magazine',
 'over',
 'another',
 "woman's",
 'shoulder.']

In [50]:
#  src shape: [25, 32]                                                                                                                                                                                      
#            sent1  sent2  sent3  sent4  ...  sent32                                                                                                                                                      
#   row 0  [  "a",  "the", "two",  "a",  ...,  "my"  ]   ← 1st word of each sentence
#   row 1  [  "dog", "cat", "men", "boy", ..., "car" ]   ← 2nd word of each sentence
#   row 2  [  "runs","sits","walk","eats",..., "is"  ]   ← 3rd word of each sentence
#   row 3  [ <EOS>, <EOS>, "fast",<EOS>, ..., "red"  ]   ← 4th word (or EOS)
#   ...
#   row 11 [ <PAD>, <PAD>, <PAD>, <PAD>, ..., <EOS>  ]   ← padding

#   So src[0] = tensor of the first token from all 32 sentences:
#   src[0]  # shape: [32] → first word of each sentence
#   src[5]  # shape: [32] → 6th word of each sentence (or PAD/EOS)

## 4. Hyperparameters (CPU-friendly)

In [59]:
# Model hyperparameters
EMBEDDING_DIM = 128
HIDDEN_DIM = 256
NUM_LAYERS = 1
DROPOUT = 0.1

# Training hyperparameters
LEARNING_RATE = 0.001
N_EPOCHS = 10
CLIP = 1.0

print('Hyperparameters set!')
print(f'  Embedding dim: {EMBEDDING_DIM}')
print(f'  Hidden dim: {HIDDEN_DIM}')
print(f'  Epochs: {N_EPOCHS}')

Hyperparameters set!
  Embedding dim: 128
  Hidden dim: 256
  Epochs: 10


## 5. Attention Implementation

**What we need to build:**

1. `Encoder` - LSTM that returns all hidden states
2. `BahdanauAttention` - Computes attention weights and context vector
3. `AttentionDecoder` - Uses attention at each step
4. `Seq2SeqAttention` - Combines everything

**Bahdanau attention formula:**
```
score(s_t, h_i) = v^T * tanh(W_h * h_i + W_s * s_t)
attention_weights = softmax(scores)
context = sum(attention_weights * encoder_outputs)
```

**Shapes to remember:**
- `encoder_outputs`: [src_len, batch, hidden_dim]
- `decoder_hidden`: [num_layers, batch, hidden_dim]
- `attention_weights`: [batch, src_len]
- `context`: [batch, hidden_dim]

In [76]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.lstm(embedded)
        return outputs, hidden, cell


In [77]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.W_h = nn.Linear(hidden_dim, hidden_dim, bias=False)  # encoder projection
        self.W_s = nn.Linear(hidden_dim, hidden_dim, bias=False)  # decoder projection
        self.v = nn.Linear(hidden_dim, 1, bias=False)             # score

    def forward(self, decoder_hidden, encoder_outputs):
        # decoder_hidden: [1, batch, hidden]
        # encoder_outputs: [src_len, batch, hidden]

        # Project encoder outputs: [src_len, batch, hidden]
        encoder_proj = self.W_h(encoder_outputs)

        # Project decoder hidden: [1, batch, hidden]
        decoder_proj = self.W_s(decoder_hidden)

        # Add with broadcasting: [src_len, batch, hidden]
        # decoder_proj broadcasts across src_len
        energy = torch.tanh(encoder_proj + decoder_proj)

        # Compute scores: [src_len, batch, 1] → [src_len, batch]
        scores = self.v(energy).squeeze(-1)

        # Softmax over source positions: [src_len, batch]
        attention_weights = F.softmax(scores, dim=0)

        # Weighted sum: [src_len, batch, 1] * [src_len, batch, hidden]
        # → sum over src_len → [batch, hidden]
        context = (attention_weights.unsqueeze(-1) * encoder_outputs).sum(dim=0)

        return context, attention_weights


In [78]:
class AttentionDecoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.attention = BahdanauAttention(hidden_dim)
        
        # LSTM input: embedding + context vector
        self.lstm = nn.LSTM(embedding_dim + hidden_dim, hidden_dim, num_layers=1)
        
        # Output layer: hidden + context → vocab
        self.fc_out = nn.Linear(hidden_dim + hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell, encoder_outputs):
        # input: [batch] (one token per sequence)
        # hidden: [1, batch, hidden]
        # cell: [1, batch, hidden]
        # encoder_outputs: [src_len, batch, hidden]
        
        # Embed input: [1, batch, emb_dim]
        input = input.unsqueeze(0)  # [batch] → [1, batch]
        embedded = self.dropout(self.embedding(input))
        
        # Compute attention: context [batch, hidden], weights [src_len, batch]
        context, attention_weights = self.attention(hidden, encoder_outputs)
        
        # Concatenate embedding + context: [1, batch, emb_dim + hidden]
        lstm_input = torch.cat([embedded, context.unsqueeze(0)], dim=2)
        
        # LSTM step: output [1, batch, hidden]
        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))
        
        # Concatenate output + context for prediction: [batch, hidden*2]
        output = torch.cat([output.squeeze(0), context], dim=1)
        
        # Predict next token: [batch, vocab_size]
        prediction = self.fc_out(output)
        
        return prediction, hidden, cell, attention_weights


In [79]:
class Seq2SeqAttention(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src: [src_len, batch]
        # trg: [trg_len, batch]

        batch_size = src.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.fc_out.out_features

        # Store outputs and attention weights
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        attentions = torch.zeros(trg_len, batch_size, src.shape[0]).to(self.device)

        # Encode source sequence
        encoder_outputs, hidden, cell = self.encoder(src)

        # First decoder input is <SOS>
        input = trg[0, :]  # [batch]

        # Decode one step at a time
        for t in range(1, trg_len):
            # Decode one step
            prediction, hidden, cell, attention_weights = self.decoder(
                input, hidden, cell, encoder_outputs
            )

            # Store prediction and attention
            outputs[t] = prediction
            attentions[t] = attention_weights.permute(1, 0)  # [batch, src_len]

            # Teacher forcing: use true token or predicted token
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = prediction.argmax(1)  # [batch]
            input = trg[t] if teacher_force else top1

        return outputs, attentions


In [ ]:
def train_epoch(model, loader, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0
    
    for src, trg in loader:
        src, trg = src.to(device), trg.to(device)

        optimizer.zero_grad()

        # Forward pass
        outputs, _ = model(src, trg)

        # outputs: [trg_len, batch, vocab_size]
        # trg: [trg_len, batch]

        # Reshape for loss: skip <SOS> at position 0
        output_dim = outputs.shape[-1]
        outputs = outputs[1:].view(-1, output_dim)  # [(trg_len-1)*batch, vocab_size]
        trg = trg[1:].view(-1)                       # [(trg_len-1)*batch]

        loss = criterion(outputs, trg)
        loss.backward()

        # Clip gradients to prevent exploding
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()
        epoch_loss += loss.item()

    return epoch_loss / len(loader)


def evaluate(model, loader, criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for src, trg in loader:
            src, trg = src.to(device), trg.to(device)

            # No teacher forcing during evaluation
            outputs, _ = model(src, trg, teacher_forcing_ratio=0)

            output_dim = outputs.shape[-1]
            outputs = outputs[1:].view(-1, output_dim)
            trg = trg[1:].view(-1)

            loss = criterion(outputs, trg)
            epoch_loss += loss.item()

    return epoch_loss / len(loader)


# Initialize model
encoder = Encoder(src_vocab.n_words, EMBEDDING_DIM, HIDDEN_DIM, DROPOUT)
decoder = AttentionDecoder(trg_vocab.n_words, EMBEDDING_DIM, HIDDEN_DIM, DROPOUT)
model = Seq2SeqAttention(encoder, decoder, device).to(device)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # ignore <PAD>

# Training loop
train_losses = []
val_losses = []

print('Training started
...\n')

for epoch in range(N_EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, CLIP)
    val_loss = evaluate(model, val_loader, criterion)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f'Epoch {epoch+1:02} | Train Loss: {train_loss:.3f} | Val Loss: {val_loss:.3f}')

print('\nTraining complete!')


Training started...

Epoch 01 | Train Loss: 5.169 | Val Loss: 4.646
Epoch 02 | Train Loss: 4.438 | Val Loss: 4.280
Epoch 03 | Train Loss: 3.935 | Val Loss: 3.987
Epoch 04 | Train Loss: 3.485 | Val Loss: 3.810
Epoch 05 | Train Loss: 3.113 | Val Loss: 3.659
Epoch 06 | Train Loss: 2.762 | Val Loss: 3.623
Epoch 07 | Train Loss: 2.475 | Val Loss: 3.559
